# Oracle Subregion Prediction (WT / TC / ET)

Trains separate oracles for WT, TC, and ET Dice prediction; compares per-subregion performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import json
import pickle as pkl
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, median_absolute_error

print('Libraries loaded.')

## Load Performance Data

In [ ]:
def load_unet_result(path):
    df = pd.read_csv(path, index_col='Unnamed: 0')
    df.index = [idx.split('-seg')[0]  # strip '-seg' suffix from U-Net inference output filenames for idx in df.index]
    df.drop(['WT jaccard', 'TC jaccard', 'ET jaccard']  # Dice is the primary metric; Jaccard unused, axis=1, inplace=True)
    return df

def read_radiomics_results(analysis_type, location):
    path = f'../../Results/Analysis_Results/Radiomics/{location}/{analysis_type}.pkl'
    with open(path, 'rb') as f:
        return pkl.load(f)

def build_feature_df(feature_list, analysis_type, location):
    raw = read_radiomics_results(analysis_type, location)
    frames = []
    for feat_name, patient_dict in raw.items():
        feat_df = pd.DataFrame.from_dict(patient_dict, orient='index').astype(float)
        feat_df.columns = [f'{feat_name}_{col}_{analysis_type}' for col in feat_df.columns]
        frames.append(feat_df)
    df = pd.concat(frames, axis=1)
    available = [f for f in feature_list if f in df.columns]
    return df[available]

performance_df = load_unet_result('../../Results/Result/Vanilla_Unet/Unet_test_dice.csv')
print(f'Performance df: {performance_df.shape}')
print('Dice distributions:')
for col in ['WT dice', 'TC dice', 'ET dice']:
    s = performance_df[col]
    print(f'  {col}: mean={s.mean():.3f}, std={s.std():.3f}, '
          f'min={s.min():.3f}, max={s.max():.3f}')

## Build 258-Feature Oracle Matrix
Exact same feature set as `Oracle Model with HPO.ipynb`.

In [ ]:
location = 'Tumor_WT'

shape_features = [
    "original_shape_Elongation_flair_shape", "original_shape_Flatness_flair_shape",
    "original_shape_LeastAxisLength_flair_shape", "original_shape_MajorAxisLength_flair_shape",
    "original_shape_MinorAxisLength_flair_shape", "original_shape_Sphericity_flair_shape",
]
size_features = [
    "diagnostics_Mask-original_VolumeNum_flair_size", "original_shape_MeshVolume_flair_size",
    "original_shape_SurfaceArea_flair_size", "original_shape_SurfaceVolumeRatio_flair_size",
]
intensity_features = [
    "diagnostics_Image-original_Mean_flair_intensity", "diagnostics_Image-original_Mean_t2_intensity",
    "diagnostics_Image-original_Mean_t1_intensity",    "diagnostics_Image-original_Mean_t1ce_intensity",
    "diagnostics_Image-original_Maximum_flair_intensity", "diagnostics_Image-original_Maximum_t2_intensity",
    "diagnostics_Image-original_Maximum_t1_intensity",    "diagnostics_Image-original_Maximum_t1ce_intensity",
]
firstorder_features = [
    "original_firstorder_Energy_flair_firstorder",    "original_firstorder_Energy_t2_firstorder",
    "original_firstorder_Energy_t1_firstorder",       "original_firstorder_Energy_t1ce_firstorder",
    "original_firstorder_Entropy_flair_firstorder",   "original_firstorder_Entropy_t2_firstorder",
    "original_firstorder_Entropy_t1_firstorder",      "original_firstorder_Entropy_t1ce_firstorder",
    "original_firstorder_Kurtosis_flair_firstorder",  "original_firstorder_Kurtosis_t2_firstorder",
    "original_firstorder_Kurtosis_t1_firstorder",     "original_firstorder_Kurtosis_t1ce_firstorder",
    "original_firstorder_Skewness_flair_firstorder",  "original_firstorder_Skewness_t2_firstorder",
    "original_firstorder_Skewness_t1_firstorder",     "original_firstorder_Skewness_t1ce_firstorder",
    "original_firstorder_Uniformity_flair_firstorder","original_firstorder_Uniformity_t2_firstorder",
    "original_firstorder_Uniformity_t1_firstorder",   "original_firstorder_Uniformity_t1ce_firstorder",
]
ngtdm_features = [
    "original_ngtdm_Busyness_flair_ngtdm_10",    "original_ngtdm_Busyness_t2_ngtdm_10",
    "original_ngtdm_Busyness_t1_ngtdm_10",       "original_ngtdm_Busyness_t1ce_ngtdm_10",
    "original_ngtdm_Coarseness_flair_ngtdm_10",  "original_ngtdm_Coarseness_t2_ngtdm_10",
    "original_ngtdm_Coarseness_t1_ngtdm_10",     "original_ngtdm_Coarseness_t1ce_ngtdm_10",
    "original_ngtdm_Complexity_flair_ngtdm_10",  "original_ngtdm_Complexity_t2_ngtdm_10",
    "original_ngtdm_Complexity_t1_ngtdm_10",     "original_ngtdm_Complexity_t1ce_ngtdm_10",
    "original_ngtdm_Contrast_flair_ngtdm_10",    "original_ngtdm_Contrast_t2_ngtdm_10",
    "original_ngtdm_Contrast_t1_ngtdm_10",       "original_ngtdm_Contrast_t1ce_ngtdm_10",
    "original_ngtdm_Strength_flair_ngtdm_10",    "original_ngtdm_Strength_t2_ngtdm_10",
    "original_ngtdm_Strength_t1_ngtdm_10",       "original_ngtdm_Strength_t1ce_ngtdm_10",
]
glcm_features = [
    "original_glcm_Autocorrelation_flair_glcm_10",     "original_glcm_Autocorrelation_t2_glcm_10",
    "original_glcm_Autocorrelation_t1_glcm_10",        "original_glcm_Autocorrelation_t1ce_glcm_10",
    "original_glcm_ClusterProminence_flair_glcm_10",   "original_glcm_ClusterProminence_t2_glcm_10",
    "original_glcm_ClusterProminence_t1_glcm_10",      "original_glcm_ClusterProminence_t1ce_glcm_10",
    "original_glcm_ClusterShade_flair_glcm_10",        "original_glcm_ClusterShade_t2_glcm_10",
    "original_glcm_ClusterShade_t1_glcm_10",           "original_glcm_ClusterShade_t1ce_glcm_10",
    "original_glcm_ClusterTendency_flair_glcm_10",     "original_glcm_ClusterTendency_t2_glcm_10",
    "original_glcm_ClusterTendency_t1_glcm_10",        "original_glcm_ClusterTendency_t1ce_glcm_10",
    "original_glcm_Contrast_flair_glcm_10",            "original_glcm_Contrast_t2_glcm_10",
    "original_glcm_Contrast_t1_glcm_10",               "original_glcm_Contrast_t1ce_glcm_10",
    "original_glcm_Correlation_flair_glcm_10",         "original_glcm_Correlation_t2_glcm_10",
    "original_glcm_Correlation_t1_glcm_10",            "original_glcm_Correlation_t1ce_glcm_10",
    "original_glcm_JointAverage_flair_glcm_10",        "original_glcm_JointAverage_t2_glcm_10",
    "original_glcm_JointAverage_t1_glcm_10",           "original_glcm_JointAverage_t1ce_glcm_10",
    "original_glcm_JointEnergy_flair_glcm_10",         "original_glcm_JointEnergy_t2_glcm_10",
    "original_glcm_JointEnergy_t1_glcm_10",            "original_glcm_JointEnergy_t1ce_glcm_10",
    "original_glcm_JointEntropy_flair_glcm_10",        "original_glcm_JointEntropy_t2_glcm_10",
    "original_glcm_JointEntropy_t1_glcm_10",           "original_glcm_JointEntropy_t1ce_glcm_10",
    "original_glcm_MCC_flair_glcm_10",                 "original_glcm_MCC_t2_glcm_10",
    "original_glcm_MCC_t1_glcm_10",                    "original_glcm_MCC_t1ce_glcm_10",
]
gldm_features = [
    "original_gldm_DependenceNonUniformity_flair_gldm_10",  "original_gldm_DependenceNonUniformity_t2_gldm_10",
    "original_gldm_DependenceNonUniformity_t1_gldm_10",     "original_gldm_DependenceNonUniformity_t1ce_gldm_10",
    "original_gldm_GrayLevelNonUniformity_flair_gldm_10",   "original_gldm_GrayLevelNonUniformity_t2_gldm_10",
    "original_gldm_GrayLevelNonUniformity_t1_gldm_10",      "original_gldm_GrayLevelNonUniformity_t1ce_gldm_10",
    "original_gldm_GrayLevelVariance_flair_gldm_10",        "original_gldm_GrayLevelVariance_t2_gldm_10",
    "original_gldm_GrayLevelVariance_t1_gldm_10",           "original_gldm_GrayLevelVariance_t1ce_gldm_10",
    "original_gldm_HighGrayLevelEmphasis_flair_gldm_10",    "original_gldm_HighGrayLevelEmphasis_t2_gldm_10",
    "original_gldm_HighGrayLevelEmphasis_t1_gldm_10",       "original_gldm_HighGrayLevelEmphasis_t1ce_gldm_10",
    "original_gldm_LargeDependenceEmphasis_flair_gldm_10",  "original_gldm_LargeDependenceEmphasis_t2_gldm_10",
    "original_gldm_LargeDependenceEmphasis_t1_gldm_10",     "original_gldm_LargeDependenceEmphasis_t1ce_gldm_10",
    "original_gldm_LowGrayLevelEmphasis_flair_gldm_10",     "original_gldm_LowGrayLevelEmphasis_t2_gldm_10",
    "original_gldm_LowGrayLevelEmphasis_t1_gldm_10",        "original_gldm_LowGrayLevelEmphasis_t1ce_gldm_10",
    "original_gldm_SmallDependenceEmphasis_flair_gldm_10",  "original_gldm_SmallDependenceEmphasis_t2_gldm_10",
    "original_gldm_SmallDependenceEmphasis_t1_gldm_10",     "original_gldm_SmallDependenceEmphasis_t1ce_gldm_10",
]
glrlm_features = [
    "original_glrlm_GrayLevelNonUniformity_flair_glrlm",           "original_glrlm_GrayLevelNonUniformity_t2_glrlm",
    "original_glrlm_GrayLevelNonUniformity_t1_glrlm",              "original_glrlm_GrayLevelNonUniformity_t1ce_glrlm",
    "original_glrlm_LongRunEmphasis_flair_glrlm",                  "original_glrlm_LongRunEmphasis_t2_glrlm",
    "original_glrlm_LongRunEmphasis_t1_glrlm",                     "original_glrlm_LongRunEmphasis_t1ce_glrlm",
    "original_glrlm_LongRunHighGrayLevelEmphasis_flair_glrlm",     "original_glrlm_LongRunHighGrayLevelEmphasis_t2_glrlm",
    "original_glrlm_LongRunHighGrayLevelEmphasis_t1_glrlm",        "original_glrlm_LongRunHighGrayLevelEmphasis_t1ce_glrlm",
    "original_glrlm_LongRunLowGrayLevelEmphasis_flair_glrlm",      "original_glrlm_LongRunLowGrayLevelEmphasis_t2_glrlm",
    "original_glrlm_LongRunLowGrayLevelEmphasis_t1_glrlm",         "original_glrlm_LongRunLowGrayLevelEmphasis_t1ce_glrlm",
    "original_glrlm_LowGrayLevelRunEmphasis_flair_glrlm",          "original_glrlm_LowGrayLevelRunEmphasis_t2_glrlm",
    "original_glrlm_LowGrayLevelRunEmphasis_t1_glrlm",             "original_glrlm_LowGrayLevelRunEmphasis_t1ce_glrlm",
    "original_glrlm_RunLengthNonUniformity_flair_glrlm",           "original_glrlm_RunLengthNonUniformity_t2_glrlm",
    "original_glrlm_RunLengthNonUniformity_t1_glrlm",              "original_glrlm_RunLengthNonUniformity_t1ce_glrlm",
    "original_glrlm_RunPercentage_flair_glrlm",                    "original_glrlm_RunPercentage_t2_glrlm",
    "original_glrlm_RunPercentage_t1_glrlm",                       "original_glrlm_RunPercentage_t1ce_glrlm",
    "original_glrlm_ShortRunEmphasis_flair_glrlm",                 "original_glrlm_ShortRunEmphasis_t2_glrlm",
    "original_glrlm_ShortRunEmphasis_t1_glrlm",                    "original_glrlm_ShortRunEmphasis_t1ce_glrlm",
    "original_glrlm_ShortRunHighGrayLevelEmphasis_flair_glrlm",    "original_glrlm_ShortRunHighGrayLevelEmphasis_t2_glrlm",
    "original_glrlm_ShortRunHighGrayLevelEmphasis_t1_glrlm",       "original_glrlm_ShortRunHighGrayLevelEmphasis_t1ce_glrlm",
    "original_glrlm_ShortRunLowGrayLevelEmphasis_flair_glrlm",     "original_glrlm_ShortRunLowGrayLevelEmphasis_t2_glrlm",
    "original_glrlm_ShortRunLowGrayLevelEmphasis_t1_glrlm",        "original_glrlm_ShortRunLowGrayLevelEmphasis_t1ce_glrlm",
]
glszm_features = [
    "original_glszm_LargeAreaEmphasis_flair_glszm",       "original_glszm_LargeAreaEmphasis_t2_glszm",
    "original_glszm_LargeAreaEmphasis_t1_glszm",          "original_glszm_LargeAreaEmphasis_t1ce_glszm",
    "original_glszm_SizeZoneNonUniformity_flair_glszm",   "original_glszm_SizeZoneNonUniformity_t2_glszm",
    "original_glszm_SizeZoneNonUniformity_t1_glszm",      "original_glszm_SizeZoneNonUniformity_t1ce_glszm",
    "original_glszm_SmallAreaEmphasis_flair_glszm",       "original_glszm_SmallAreaEmphasis_t2_glszm",
    "original_glszm_SmallAreaEmphasis_t1_glszm",          "original_glszm_SmallAreaEmphasis_t1ce_glszm",
    "original_glszm_ZoneEntropy_flair_glszm",             "original_glszm_ZoneEntropy_t2_glszm",
    "original_glszm_ZoneEntropy_t1_glszm",                "original_glszm_ZoneEntropy_t1ce_glszm",
]
volume_features = ['ED', 'ET', 'NCR', 'WT_volume', 'TC_volume', 'ET_volume',
                   'TC_WT_ratio', 'ET_WT_ratio', 'ET_TC_ratio']
curvature_features = ['mean_gaussian_curvature', 'std_gaussian_curvature',
                      'pos', 'neg', 'pos_count', 'neg_count']

# Build feature DataFrames
shape_df      = build_feature_df(shape_features,      'shape',      location)
size_df       = build_feature_df(size_features,        'size',       location)
intensity_df  = build_feature_df(intensity_features,   'intensity',  location)
firstorder_df = build_feature_df(firstorder_features,  'firstorder', location)
ngtdm_df      = build_feature_df(ngtdm_features,       'ngtdm_10',   location)
glcm_df       = build_feature_df(glcm_features,        'glcm_10',    location)
gldm_df       = build_feature_df(gldm_features,        'gldm_10',    location)
glrlm_df      = build_feature_df(glrlm_features,       'glrlm',      location)
glszm_df      = build_feature_df(glszm_features,       'glszm',      location)

volume_df = pd.read_csv('../../Results/Analysis_Results/volume/GLI-Tumor_volumns.csv',
                         index_col='Unnamed: 0')[volume_features]
prob_df   = pd.read_csv('../../Results/Analysis_Results/probability/Probability_Tumor_boundary.csv',
                         index_col='Unnamed: 0')
curv_df   = pd.read_csv('../../Results/Analysis_Results/curverature/curverature.csv',
                         index_col='Unnamed: 0')[curvature_features]
sal_df    = pd.read_csv('../../Results/Analysis_Results/Saliency/Saliency.csv',
                         index_col='Unnamed: 0')

# Merge all features + performance
dfs = [shape_df, size_df, intensity_df, firstorder_df, ngtdm_df,
       glcm_df, gldm_df, glrlm_df, glszm_df,
       volume_df, prob_df, curv_df, sal_df, performance_df]

full_df = dfs[0]
for d in dfs[1:]:
    full_df = full_df.join(d, how='inner')

full_df = full_df.dropna()
print(f'Full matrix after dropna: {full_df.shape}')

dice_cols = ['WT dice', 'TC dice', 'ET dice']
feat_cols = [c for c in full_df.columns if c not in dice_cols]
X = full_df[feat_cols]
print(f'Feature matrix: {X.shape}')

## Train Oracle for WT, TC, ET — 5-Fold CV

GBR hyperparameters are set to the best values found by HPO for WT Dice  
(learning_rate=0.10, n_estimators=150, max_depth=5) to keep runtime manageable.  
The same params are used for TC and ET to ensure a fair comparison.

In [ ]:
GBR_PARAMS = dict(
    n_estimators=150,  # tuned via Bayesian HPO (see Oracle_Model_with_HPO.ipynb),
    learning_rate=0.10,
    max_depth=5,
    subsample=0.75,
    min_samples_split=3,
    min_samples_leaf=2,
    random_state=42,
)

TARGETS = {
    'WT': ('WT dice', 0.91),
    'TC': ('TC dice', 0.86),
    'ET': ('ET dice', 0.85),
}

results = {}       # per-target metrics
preds_all = {}     # per-target full predictions (for scatter plots)
importances = {}   # per-target feature importances (last fold)

kf = KFold(n_splits=5  # 5-fold CV — matches oracle evaluation protocol, shuffle=True, random_state=42)

for target_name, (target_col, threshold) in TARGETS.items():
    y = full_df[target_col]
    mae_scores, r2_scores, mse_scores, medae_scores = [], [], [], []
    y_true_all, y_pred_all = [], []
    last_model = None

    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_train)
        X_te = scaler.transform(X_test)

        model = GradientBoostingRegressor(**GBR_PARAMS)
        model.fit(X_tr, y_train)
        y_pred = model.predict(X_te)

        mae_scores.append(mean_absolute_error(y_test, y_pred))
        r2_scores.append(r2_score(y_test, y_pred))
        mse_scores.append(mean_squared_error(y_test, y_pred))
        medae_scores.append(median_absolute_error(y_test, y_pred))
        y_true_all.extend(y_test.tolist())
        y_pred_all.extend(y_pred.tolist())
        last_model = model

    results[target_name] = {
        'MAE':   (np.mean(mae_scores),   np.std(mae_scores)),
        'R2':    (np.mean(r2_scores),     np.std(r2_scores)),
        'MSE':   (np.mean(mse_scores),    np.std(mse_scores)),
        'MedAE': (np.mean(medae_scores),  np.std(medae_scores)),
        'threshold': threshold,
    }
    preds_all[target_name] = (np.array(y_true_all), np.array(y_pred_all))
    importances[target_name] = last_model.feature_importances_

    print(f'{target_name}  MAE={np.mean(mae_scores):.3f}±{np.std(mae_scores):.3f}  '
          f'MedAE={np.mean(medae_scores):.3f}±{np.std(medae_scores):.3f}  '
          f'R2={np.mean(r2_scores):.3f}±{np.std(r2_scores):.3f}  '
          f'MSE={np.mean(mse_scores):.4f}±{np.std(mse_scores):.4f}')

## Figure 1 — Performance Comparison (MAE, R², MedAE)

In [ ]:
COLORS = {'WT': 'steelblue', 'TC': 'darkorange', 'ET': 'seagreen'}
targets = list(TARGETS.keys())

metrics = [
    ('MAE',   'Mean Absolute Error (MAE)', 'lower is better'),
    ('R2',    'R\u00b2 Score',             'higher is better'),
    ('MedAE', 'Median Absolute Error',     'lower is better'),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Oracle GBR Performance by Tumor Subregion Target', fontsize=12)

for ax, (metric, label, note) in zip(axes, metrics):
    means = [results[t][metric][0] for t in targets]
    stds  = [results[t][metric][1] for t in targets]
    colors = [COLORS[t] for t in targets]
    bars = ax.bar(targets, means, yerr=stds, capsize=6,
                  color=colors, edgecolor='white', width=0.5)
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2,
                m + s + 0.003, f'{m:.3f}', ha='center', fontsize=9)
    ax.set_ylabel(label)
    ax.set_title(f'{label}\n({note})', fontsize=9)
    ax.set_xticks(range(len(targets)))
    ax.set_xticklabels(targets, fontsize=11)

plt.tight_layout()
plt.savefig('../../Results/Figures/oracle_subregion_performance.pdf', bbox_inches='tight')
plt.show()
print('Saved: ../../Results/Figures/oracle_subregion_performance.pdf')

## Figure 2 — Predicted vs Actual Dice Scatter Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Oracle: Predicted vs Actual Dice Score (5-Fold CV)', fontsize=12)

for ax, target_name in zip(axes, targets):
    y_true, y_pred = preds_all[target_name]
    thresh = TARGETS[target_name][1]
    mae  = results[target_name]['MAE'][0]
    r2   = results[target_name]['R2'][0]

    ax.scatter(y_true, y_pred, alpha=0.25, s=8, color=COLORS[target_name])
    lim = [min(y_true.min(), y_pred.min()) - 0.02,
           max(y_true.max(), y_pred.max()) + 0.02]
    ax.plot(lim, lim, 'k--', linewidth=1, label='Perfect')
    ax.axvline(thresh, color='red', linestyle=':', linewidth=1,
               label=f'Threshold ({thresh})')
    ax.axhline(thresh, color='red', linestyle=':', linewidth=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel(f'Actual {target_name} Dice')
    ax.set_ylabel(f'Predicted {target_name} Dice')
    ax.set_title(f'{target_name} Dice\nMAE={mae:.3f}, R\u00b2={r2:.3f}')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('../../Results/Figures/oracle_subregion_scatter.pdf', bbox_inches='tight')
plt.show()
print('Saved: ../../Results/Figures/oracle_subregion_scatter.pdf')

## Figure 3 — Top-20 Feature Importance per Subregion

In [ ]:
def short_name(feat):
    parts = feat.split('_')
    # e.g. original_glcm_Contrast_t2_glcm_10 -> glcm_Contrast_t2
    if len(parts) >= 4:
        return '_'.join(parts[1:4])
    return feat

fig, axes = plt.subplots(1, 3, figsize=(18, 8))
fig.suptitle('Top-20 Feature Importances by Oracle Target', fontsize=12)

for ax, target_name in zip(axes, targets):
    imp = importances[target_name]
    imp_df = pd.Series(imp, index=feat_cols).sort_values(ascending=False).head(20)
    short = [short_name(f) for f in imp_df.index]
    ax.barh(range(len(imp_df)), imp_df.values[::-1],
            color=COLORS[target_name], edgecolor='white')
    ax.set_yticks(range(len(imp_df)))
    ax.set_yticklabels(short[::-1], fontsize=7)
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'{target_name} Dice Oracle\nTop-20 Features (last fold)')

plt.tight_layout()
plt.savefig('../../Results/Figures/oracle_subregion_feature_importance.pdf', bbox_inches='tight')
plt.show()
print('Saved: ../../Results/Figures/oracle_subregion_feature_importance.pdf')

## Figure 4 — Prediction Error Distribution
Error = predicted - actual Dice. Shows systematic bias and spread per subregion.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Prediction Error Distribution (Predicted - Actual Dice)', fontsize=12)

for ax, target_name in zip(axes, targets):
    y_true, y_pred = preds_all[target_name]
    errors = y_pred - y_true
    ax.hist(errors, bins=50, color=COLORS[target_name], edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', linewidth=1.5, linestyle='--')
    ax.axvline(errors.mean(), color='red', linewidth=1.2, linestyle='-',
               label=f'Mean={errors.mean():.3f}')
    ax.set_xlabel('Prediction Error')
    ax.set_ylabel('Count')
    ax.set_title(f'{target_name} Dice\nstd={errors.std():.3f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../../Results/Figures/oracle_subregion_error_dist.pdf', bbox_inches='tight')
plt.show()
print('Saved: ../../Results/Figures/oracle_subregion_error_dist.pdf')

## Classification Accuracy at Threshold
How well does the Oracle classify cases as good/poor when thresholding its continuous predictions?

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print('Binary classification accuracy (threshold applied to predicted Dice)\n')
clf_results = {}
for target_name, (target_col, thresh) in TARGETS.items():
    y_true, y_pred = preds_all[target_name]
    true_bad  = (y_true < thresh).astype(int)
    pred_bad  = (y_pred < thresh).astype(int)

    cm = confusion_matrix(true_bad, pred_bad)
    report = classification_report(true_bad, pred_bad,
                                   target_names=['good', 'poor'],
                                   output_dict=True)
    clf_results[target_name] = report

    print(f'--- {target_name} (threshold={thresh}) ---')
    print(classification_report(true_bad, pred_bad,
                                target_names=['good', 'poor']))
    print(f'Confusion matrix:\n{cm}\n')

## Save Results

In [ ]:
output = {
    'model': 'GradientBoostingRegressor',
    'hyperparameters': GBR_PARAMS,
    'cv': '5-fold KFold (shuffle=True, random_state=42)',
    'n_features': X.shape[1],
    'n_samples': X.shape[0],
    'regression_results': {
        target: {
            'MAE_mean':   round(v['MAE'][0],   4),
            'MAE_std':    round(v['MAE'][1],    4),
            'R2_mean':    round(v['R2'][0],     4),
            'R2_std':     round(v['R2'][1],     4),
            'MSE_mean':   round(v['MSE'][0],    5),
            'MSE_std':    round(v['MSE'][1],    5),
            'MedAE_mean': round(v['MedAE'][0],  4),
            'MedAE_std':  round(v['MedAE'][1],  4),
            'threshold':  v['threshold'],
        }
        for target, v in results.items()
    },
    'classification_results': {
        t: {
            'precision_poor': round(clf_results[t]['poor']['precision'], 3),
            'recall_poor':    round(clf_results[t]['poor']['recall'],    3),
            'f1_poor':        round(clf_results[t]['poor']['f1-score'],  3),
            'precision_good': round(clf_results[t]['good']['precision'], 3),
            'recall_good':    round(clf_results[t]['good']['recall'],    3),
            'f1_good':        round(clf_results[t]['good']['f1-score'],  3),
        }
        for t in TARGETS
    },
}

with open('../../Results/Json_summary/oracle_subregion_results.json', 'w') as f:
    json.dump(output, f, indent=4)

print('Saved: ../../Results/Json_summary/oracle_subregion_results.json')
print()
print('=== FINAL SUMMARY FOR PAPER ===')
SEP = '-' * 65
print(f'{"Target":6s}  {"MAE":>16}  {"MedAE":>16}  {"R2":>14}')
print(SEP)
for t in targets:
    v = results[t]
    print(f'{t:6s}  {v["MAE"][0]:.3f} +/- {v["MAE"][1]:.3f}'
          f'  {v["MedAE"][0]:.3f} +/- {v["MedAE"][1]:.3f}'
          f'  {v["R2"][0]:.3f} +/- {v["R2"][1]:.3f}')
print()
print('Binary classification (threshold applied to Oracle predictions):')
print(f'{"Target":6s}  {"Recall(poor)":>14}  {"F1(poor)":>10}  {"F1(good)":>10}')
print(SEP)
for t in targets:
    c = clf_results[t]
    print(f'{t:6s}  {c["poor"]["recall"]:>14.3f}  {c["poor"]["f1-score"]:>10.3f}  {c["good"]["f1-score"]:>10.3f}')